# 02 - Feature audit

Audit of the version-1 hostname features using **TRAIN data only** (the test set is never touched here). The goal is to find features that are suspiciously predictive on their own, which usually means a dataset shortcut rather than a real phishing signal.

> **Safety:** hostnames are only handled as text. No DNS, WHOIS or any network call is made.

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from src.features import extract_features_df

pd.set_option("display.width", 200)
train = pd.read_csv("../data/processed/train.csv")
train["cls"] = train["label"].map({1: "phishing", 0: "legitimate"})
print(train.shape)
train.head()

(175228, 4)


,hostname,registered_domain,label,cls
0,southbankmosaics.com,southbankmosaics.com,0,legitimate
1,uni-mainz.de,uni-mainz.de,0,legitimate
2,voicefmradio.co.uk,voicefmradio.co.uk,0,legitimate
3,sfnmjournal.com,sfnmjournal.com,0,legitimate
4,rewildingargentina.org,rewildingargentina.org,0,legitimate


## 1. Compute features
Runs `extract_features` (pure Python, offline) on every training hostname.

In [2]:
X = extract_features_df(train["hostname"])
y = train["label"]
groups = train["registered_domain"]
X.describe().T[["mean", "min", "max"]].round(3)

,mean,min,max
hostname_length,18.545,4.0,110.000
num_labels,2.397,2.0,12.000
num_subdomain_levels,0.302,0.0,10.000
num_digits,0.864,0.0,44.000
digit_ratio,0.031,0.0,0.886
num_hyphens,0.271,0.0,26.000
longest_label_length,12.167,2.0,63.000
domain_entropy,2.767,-0.0,4.500
vowel_ratio,0.352,0.0,1.000
is_ip,0.001,0.0,1.000


## 2. Feature mean per class
The mean of each feature for legitimate vs phishing hostnames. For 0/1 features the mean is the fraction of hostnames where the feature is on. Big gaps are candidates for strong signals (or shortcuts).

In [3]:
means = X.groupby(train["cls"]).mean().T.round(3)
means["ratio phish/legit"] = (means["phishing"] / means["legitimate"].replace(0, float("nan"))).round(2)
means

cls,legitimate,phishing,ratio phish/legit
hostname_length,15.232,23.847,1.57
num_labels,2.162,2.772,1.28
num_subdomain_levels,0.030,0.736,24.53
num_digits,0.051,2.167,42.49
digit_ratio,0.004,0.074,18.50
num_hyphens,0.084,0.571,6.80
longest_label_length,10.895,14.201,1.30
domain_entropy,2.818,2.684,0.95
vowel_ratio,0.366,0.328,0.90
is_ip,0.000,0.002,NaN


## 3. Single-feature predictive power (grouped CV)
For each feature alone, a small decision tree (`max_depth=3`) is evaluated with 5-fold `StratifiedGroupKFold` **grouped by registered domain**, so a domain never sits in both the training and validation part of a fold. The score is the mean ROC-AUC.

A single feature with a high AUC is flagged: **above 0.90 = possible shortcut**. It does not mean the feature is useless, only that it deserves scrutiny before trusting a model that relies on it.

In [4]:
# Build the folds ONCE (splitting 140k groups is slow) and reuse them for every feature.
folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(X, y, groups))
rows = []
for col in X.columns:
    tree = DecisionTreeClassifier(max_depth=3, random_state=42)
    scores = cross_val_score(tree, X[[col]], y, cv=folds, scoring="roc_auc")
    rows.append({"feature": col, "auc_mean": scores.mean(), "auc_std": scores.std()})
auc = pd.DataFrame(rows).sort_values("auc_mean", ascending=False).reset_index(drop=True)
auc["flag"] = auc["auc_mean"].map(lambda a: "POSSIBLE SHORTCUT (>0.90)" if a > 0.90 else "")
auc.round(4)

,feature,auc_mean,auc_std,flag
0,num_subdomain_levels,0.7633,0.0013,
1,num_labels,0.7221,0.0064,
2,hostname_length,0.7060,0.0218,
3,num_digits,0.6790,0.0166,
4,digit_ratio,0.6773,0.0161,
5,num_hyphens,0.6284,0.0134,
6,is_free_hosting,0.6021,0.0722,
7,vowel_ratio,0.5824,0.0337,
8,longest_label_length,0.5822,0.0210,
9,domain_entropy,0.5715,0.0466,


## 4. Subdomains and free hosting per class
Two structural features that look innocent but may mostly reflect *where the data was collected*: the share of hostnames with at least one subdomain level (after stripping `www.`), and the share on a known free-hosting / site-builder service. If one class has almost none of either, the model can learn a collection artifact instead of phishing behaviour.

In [5]:
share = pd.DataFrame({
    "% with >=1 subdomain level": X["num_subdomain_levels"] >= 1,
    "% on free hosting": X["is_free_hosting"] == 1,
}).groupby(train["cls"]).mean().mul(100).round(2)
share["n hostnames"] = train["cls"].value_counts()
share

,% with >=1 subdomain level,% on free hosting,n hostnames
cls,,,
legitimate,2.87,0.00,107833
phishing,55.21,20.42,67395


## 5. Extra context: distribution of subdomain levels per class
Normalized counts, to see how deep the subdomain structure goes in each class.

In [6]:
pd.crosstab(X["num_subdomain_levels"].clip(upper=4), train["cls"], normalize="columns").mul(100).round(2)

cls,legitimate,phishing
num_subdomain_levels,,
0,97.13,44.79
1,2.69,40.80
2,0.18,11.55
3,0.00,1.83
4,0.00,1.03
